In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "591ab4e3a3e575d95d805a92550ef076fd65a824"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/591ab4e3a3e575d95d805a92550ef076fd65a824/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Denoising Diffusion Probabilistic Models

The ladder of the previous section is a collection of separate perturbed
distributions, and the sampler that walks down it has a step size and a
number of steps per level that must be set by hand. A diffusion model
replaces the ladder by a single Markov process that turns data into noise in
$T$ small steps, and it learns the reversal of that process as a
latent-variable model, whose training objective derives from a variational
bound on the likelihood in the manner of that section. The construction is due to
@sohl2015deep; @ho2020denoising showed that each term of the
bound becomes a weighted noise-prediction regression, the same regression
as denoising score matching, and that the resulting samples rival those of
adversarial models. This section derives the forward process, the bound, the
simplification, and the sampler, and checks the marginal, the noise
predictor, and the sampler on the running example. The presentation follows
lecture 13 of @Kuleshov.2023; the monograph of @Lai.Song.Kim.ea.2025 develops the same derivation and
its alternatives at greater length.

In [ ]:
%matplotlib inline
from d2l import torch as d2l
import math
import torch
from torch import nn

## The Forward Process

Fix a number of steps $T$ and a *variance schedule* $\beta_1, \ldots,
\beta_T \in (0, 1)$. The **forward process** starts from a data point
$\mathbf{x}_0 \sim q(\mathbf{x}_0)$, the data distribution ($q$ denotes
every distribution of the forward process), and applies $T$ Gaussian
transitions,

$$
q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\big(\mathbf{x}_t;\ \sqrt{1 - \beta_t}\, \mathbf{x}_{t-1},\ \beta_t I\big),
\qquad
q(\mathbf{x}_{1:T} \mid \mathbf{x}_0) = \prod_{t=1}^{T} q(\mathbf{x}_t \mid \mathbf{x}_{t-1}),
$$

each of which shrinks the current state by $\sqrt{1 - \beta_t}$ and adds
noise of variance $\beta_t$. @ho2020denoising use $T = 1000$ with
$\beta_t$ increasing linearly from $10^{-4}$ to $0.02$; the two-dimensional
experiments below use $T = 200$ with a correspondingly larger range. The
process is fixed in advance and has no parameters.
the figure shows it as the upper row of arrows.

![The graphical model of a diffusion model. The forward process (top) is a fixed Markov chain of Gaussian transitions that carries data $\mathbf{x}_0$ to noise $\mathbf{x}_T$. The reverse process (bottom) is a learned Markov chain that runs the other way; its transitions are Gaussians whose means a network predicts. The shaded node $\mathbf{x}_0$ is observed; $\mathbf{x}_1, \ldots, \mathbf{x}_T$ are latent.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-diffusion-forward-reverse.svg)

The shrinking factor distinguishes the process from the noise ladder of
that section, which added noise without shrinking. If
$\mathbf{x}_{t-1}$ has identity covariance, then $\mathbf{x}_t$ has
covariance $(1 - \beta_t) I + \beta_t I = I$: the transition preserves unit
variance, so the chain moves data toward a *standard* Gaussian rather than
toward an ever wider one, and the end point does not depend on the scale of
the data once the remaining signal $\sqrt{\bar{\alpha}_T}\, \mathbf{x}_0$,
defined below, is negligible. Writing $\alpha_t = 1 - \beta_t$ (the symbol
$\alpha$ is reused here; the Langevin step size of the previous sections is
not needed again), a step is
$\mathbf{x}_t = \sqrt{\alpha_t}\, \mathbf{x}_{t-1} + \sqrt{\beta_t}\, \boldsymbol{\epsilon}_t$
with independent $\boldsymbol{\epsilon}_t \sim \mathcal{N}(\mathbf{0}, I)$.

### The Marginal in Closed Form

Composing two steps gives
$\mathbf{x}_t = \sqrt{\alpha_t \alpha_{t-1}}\, \mathbf{x}_{t-2} + \sqrt{\alpha_t \beta_{t-1}}\, \boldsymbol{\epsilon}_{t-1} + \sqrt{\beta_t}\, \boldsymbol{\epsilon}_t$.
The two noise terms are independent Gaussians, so their sum is a Gaussian
with variance $\alpha_t \beta_{t-1} + \beta_t = \alpha_t (1 - \alpha_{t-1}) + 1 - \alpha_t = 1 - \alpha_t \alpha_{t-1}$.
Repeating the argument down to $\mathbf{x}_0$, with
$\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$, gives the marginal of any step
directly from the data:

$$
q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\big(\mathbf{x}_t;\ \sqrt{\bar{\alpha}_t}\, \mathbf{x}_0,\ (1 - \bar{\alpha}_t) I\big),
\qquad\textrm{i.e.}\qquad
\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\, \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\, \boldsymbol{\epsilon},
\quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, I).
$$

The formal induction is given in that section
(the equation). The formula has two consequences. First,
training never needs to simulate the chain: any $\mathbf{x}_t$ is one
Gaussian draw away from $\mathbf{x}_0$. Second, the marginals are, up to
the rescaling by $\sqrt{\bar{\alpha}_t}$, the noise ladder of
that section. Dividing by $\sqrt{\bar{\alpha}_t}$ gives
$\mathbf{x}_t / \sqrt{\bar{\alpha}_t} = \mathbf{x}_0 + \sqrt{(1 - \bar{\alpha}_t) / \bar{\alpha}_t}\; \boldsymbol{\epsilon}$,
a perturbed copy of the data at a noise level that grows with $t$. With the
schedule of
@ho2020denoising, $\bar{\alpha}_T \approx 4 \times 10^{-5}$, so
$q(\mathbf{x}_T \mid \mathbf{x}_0)$ is a standard Gaussian to within a
scaled-down trace of $\mathbf{x}_0$, whatever $\mathbf{x}_0$ was.

The cell sets up the schedule for the running example and checks
the equation by simulation: it runs the chain on the
training set and compares the mean and variance of the resulting
$\mathbf{x}_t$ with those of a one-shot draw from the closed form. In the
code, index `t` runs from $0$ to $T - 1$ and stands for step $t + 1$ of the
text.

In [ ]:
torch.manual_seed(0)
mix = d2l.GaussianMixture(means=[[-2.5, -1.5], [2.5, -1.5], [0.0, 2.5]],
                          weights=[0.5, 0.3, 0.2], std=0.5)
data, test = mix.sample(2000), mix.sample(2000)

T = 200
beta = torch.linspace(1e-4, 0.1, T)
alpha = 1 - beta
alpha_bar = torch.cumprod(alpha, 0)
print(f'alpha_bar_T = {alpha_bar[-1]:.1e}: after T steps the data are scaled '
      f'by {alpha_bar[-1].sqrt():.3f}')

torch.manual_seed(1)
x = data.clone()
for t in range(T):  # the chain, step by step
    x = alpha[t].sqrt() * x + beta[t].sqrt() * torch.randn_like(x)
    if t + 1 in (10, 50, 200):
        one_shot = (alpha_bar[t].sqrt() * data
                    + (1 - alpha_bar[t]).sqrt() * torch.randn_like(data))
        print(f'step {t + 1:3d}: chain: mean {[round(m, 3) for m in x.mean(0).tolist()]}, '
              f'var {x.var(0).mean():.3f} | one shot: mean '
              f'{[round(m, 3) for m in one_shot.mean(0).tolist()]}, '
              f'var {one_shot.var(0).mean():.3f}')

The chain and the one-shot draw agree closely at every checkpoint, and after
two hundred steps both have mean near zero and unit variance: the mixture
has been carried very nearly to a standard Gaussian.

## The Reverse Process

Generation runs the chain backward: draw $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, I)$
and sample $\mathbf{x}_{t-1}$ from the reversal of each forward step until
$\mathbf{x}_0$ is reached. The exact reversal $q(\mathbf{x}_{t-1} \mid \mathbf{x}_t)$
is, by Bayes' rule, proportional to $q(\mathbf{x}_t \mid \mathbf{x}_{t-1})\, q(\mathbf{x}_{t-1})$,
and it involves the marginal $q(\mathbf{x}_{t-1})$, which depends on the
entire data distribution and is unavailable. Two facts make it learnable.
When the steps are small, the reversal of a Gaussian transition is itself
close to a Gaussian, a classical observation that @sohl2015deep
turned into a model; in the continuous-time limit the time reversal of the
forward diffusion is exactly a diffusion whose drift involves the score of
the marginals (that section, [@Anderson.1982]). The
**reverse process** is therefore modeled as a Markov chain of Gaussians
running from $T$ to $0$,

$$
p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T}) = p(\mathbf{x}_T) \prod_{t=1}^{T} p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t),
\qquad
p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t) = \mathcal{N}\big(\mathbf{x}_{t-1};\ \boldsymbol{\mu}_{\boldsymbol{\theta}}(\mathbf{x}_t, t),\ \sigma_t^2 I\big),
$$

with $p(\mathbf{x}_T) = \mathcal{N}(\mathbf{0}, I)$, a mean
$\boldsymbol{\mu}_{\boldsymbol{\theta}}$ computed by a network that receives
the current state and the step index, and variances $\sigma_t^2$ that we fix
in advance. Here $\sigma_t$ is the standard deviation of a reverse step, not
the perturbation level $\sigma_i$ of the previous sections; the noise level
of $\mathbf{x}_t$ is $\sqrt{1 - \bar{\alpha}_t}$.
@ho2020denoising found that $\sigma_t^2 = \beta_t$ and the posterior
variance derived below give similar results, and
@Nichol.Dhariwal.2021 later learned the variances. The intermediate
states
$\mathbf{x}_1, \ldots, \mathbf{x}_T$ are latent variables, and the model
density of a data point is the marginal
$p_{\boldsymbol{\theta}}(\mathbf{x}_0) = \int p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T})\, d\mathbf{x}_{1:T}$,
an integral over $T$ copies of the data space that is intractable.

## The Variational Bound

An intractable marginal likelihood with a tractable joint is the situation
of that section, and the same device applies: bound the
log-likelihood from below with an auxiliary distribution over the latents.
Here the auxiliary distribution is the forward process itself,
$q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)$, which is fixed rather than learned.
Multiplying and dividing by it inside the integral and applying Jensen's
inequality (that section) to the concave logarithm,

$$
\log p_{\boldsymbol{\theta}}(\mathbf{x}_0)
= \log \mathbb{E}_{q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)}\!\left[ \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)} \right]
\;\geq\;
\mathbb{E}_{q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)}\!\left[ \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)} \right]
=: -L_{\textrm{VLB}}(\mathbf{x}_0).
$$

The gap between the two sides is the Kullback--Leibler divergence
$\mathrm{KL}\big(q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)\, \|\, p_{\boldsymbol{\theta}}(\mathbf{x}_{1:T} \mid \mathbf{x}_0)\big)$,
so minimizing $L_{\textrm{VLB}}$ pushes the model likelihood up and pulls
the learned reverse process toward the true reversal of the forward one.

### Decomposition into Per-Step Terms

Both processes are Markov chains, and substituting their factorizations
turns the bound into a sum of one term per step. Writing $\mathbb{E}_q$ for
the expectation over $q(\mathbf{x}_{1:T} \mid \mathbf{x}_0)$,

$$
\begin{aligned}
L_{\textrm{VLB}}
&= \mathbb{E}_q\!\left[ -\log p(\mathbf{x}_T) - \sum_{t=1}^{T} \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t)}{q(\mathbf{x}_t \mid \mathbf{x}_{t-1})} \right] \\
&= \mathbb{E}_q\!\left[ -\log p(\mathbf{x}_T) - \sum_{t=2}^{T} \log \left( \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t)}{q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)} \cdot \frac{q(\mathbf{x}_{t-1} \mid \mathbf{x}_0)}{q(\mathbf{x}_t \mid \mathbf{x}_0)} \right) - \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_0 \mid \mathbf{x}_1)}{q(\mathbf{x}_1 \mid \mathbf{x}_0)} \right] \\
&= \mathbb{E}_q\!\left[ \log \frac{q(\mathbf{x}_T \mid \mathbf{x}_0)}{p(\mathbf{x}_T)} + \sum_{t=2}^{T} \log \frac{q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)}{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t)} - \log p_{\boldsymbol{\theta}}(\mathbf{x}_0 \mid \mathbf{x}_1) \right].
\end{aligned}
$$

The second line rewrites each forward transition with $t \geq 2$ through
Bayes' rule conditioned on $\mathbf{x}_0$,
$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = q(\mathbf{x}_t \mid \mathbf{x}_{t-1}, \mathbf{x}_0) = q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)\, q(\mathbf{x}_t \mid \mathbf{x}_0) / q(\mathbf{x}_{t-1} \mid \mathbf{x}_0)$,
where the first equality is the Markov property. The third line collapses
the telescoping product of the ratios $q(\mathbf{x}_{t-1} \mid \mathbf{x}_0) / q(\mathbf{x}_t \mid \mathbf{x}_0)$
over $t = 2, \ldots, T$ into $q(\mathbf{x}_1 \mid \mathbf{x}_0) / q(\mathbf{x}_T \mid \mathbf{x}_0)$,
whose numerator cancels the factor $q(\mathbf{x}_1 \mid \mathbf{x}_0)$ in the
$t = 1$ term. Each logarithmic ratio in the
final line is, in expectation, a Kullback--Leibler divergence between two
distributions of the same variable, so

$$
L_{\textrm{VLB}}
= \underbrace{\mathrm{KL}\big(q(\mathbf{x}_T \mid \mathbf{x}_0)\, \|\, p(\mathbf{x}_T)\big)}_{L_T}
+ \sum_{t=2}^{T} \underbrace{\mathbb{E}_q\Big[\mathrm{KL}\big(q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)\, \|\, p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t)\big)\Big]}_{L_{t-1}}
+ \underbrace{\mathbb{E}_q\big[-\log p_{\boldsymbol{\theta}}(\mathbf{x}_0 \mid \mathbf{x}_1)\big]}_{L_0}.
$$

The three kinds of terms have different roles [@ho2020denoising]. The
*prior term* $L_T$ compares the end of the forward chain with the standard
Gaussian that the reverse chain starts from; it contains no parameters and
is close to zero for a schedule that reaches $\bar{\alpha}_T \approx 0$. The
*reconstruction term* $L_0$ scores the final denoising step. The
*denoising terms* $L_{t-1}$ carry the training signal: each asks the learned
transition $p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} \mid \mathbf{x}_t)$ to
match $q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)$, the reversal
of step $t$ *given the clean point*. Conditioning on $\mathbf{x}_0$ is what
makes the target tractable: unlike $q(\mathbf{x}_{t-1} \mid \mathbf{x}_t)$,
which requires the data marginal, this posterior involves only the two
Gaussian kernels of the forward process.

### The Forward Posterior

By Bayes' rule, $q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0) \propto q(\mathbf{x}_t \mid \mathbf{x}_{t-1})\, q(\mathbf{x}_{t-1} \mid \mathbf{x}_0)$,
a product of two Gaussians in $\mathbf{x}_{t-1}$: the first has mean
$\mathbf{x}_t / \sqrt{\alpha_t}$ and variance $\beta_t / \alpha_t$ as a
function of $\mathbf{x}_{t-1}$, and the second has mean
$\sqrt{\bar{\alpha}_{t-1}}\, \mathbf{x}_0$ and variance $1 - \bar{\alpha}_{t-1}$.
Precisions of Gaussian factors add and means combine in proportion to
precision, so the posterior is Gaussian,
$q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}(\tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0), \tilde{\beta}_t I)$,
with

$$
\frac{1}{\tilde{\beta}_t} = \frac{\alpha_t}{\beta_t} + \frac{1}{1 - \bar{\alpha}_{t-1}} = \frac{1 - \bar{\alpha}_t}{\beta_t (1 - \bar{\alpha}_{t-1})},
\qquad
\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1 - \bar{\alpha}_t}\, \mathbf{x}_0 + \frac{\sqrt{\alpha_t}\, (1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t}\, \mathbf{x}_t,
$$

where the simplification of the precision uses $\alpha_t (1 - \bar{\alpha}_{t-1}) + \beta_t = 1 - \bar{\alpha}_t$
(Exercise 2 fills in the algebra). The posterior mean is a linear
combination of the clean point and the current noisy point, and its variance
$\tilde{\beta}_t = \beta_t (1 - \bar{\alpha}_{t-1}) / (1 - \bar{\alpha}_t)$
is smaller than $\beta_t$: much smaller at the first steps, where
$\tilde{\beta}_1 = 0$, and nearly equal once $\bar{\alpha}_{t-1}$ is well
below one.

## From the Bound to Noise Prediction

With the reverse variances fixed at $\sigma_t^2$, each denoising term is a
Kullback--Leibler divergence between two isotropic Gaussians whose
covariances do not depend on $\boldsymbol{\theta}$, which by
the equation is a scaled squared distance between their means
plus a constant (the constant vanishes when $\sigma_t^2 = \tilde{\beta}_t$):

$$
L_{t-1} = \mathbb{E}_q\!\left[ \frac{1}{2 \sigma_t^2} \big\| \tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0) - \boldsymbol{\mu}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) \big\|^2 \right] + \textrm{const}.
$$

The network could predict $\tilde{\boldsymbol{\mu}}_t$ directly.
@ho2020denoising found that a change of variables gives a target
that works about as well with the bound and much better with the simplified
loss below. Expressing the clean point through the noise that produced
$\mathbf{x}_t$ in the equation,
$\mathbf{x}_0 = (\mathbf{x}_t - \sqrt{1 - \bar{\alpha}_t}\, \boldsymbol{\epsilon}) / \sqrt{\bar{\alpha}_t}$,
and substituting into the equation, the coefficients of
$\mathbf{x}_t$ combine into $1 / \sqrt{\alpha_t}$ and the posterior mean
becomes

$$
\tilde{\boldsymbol{\mu}}_t = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \boldsymbol{\epsilon} \right).
$$

Given $\mathbf{x}_t$ and $t$, which the network receives as inputs,
everything in this expression is known except $\boldsymbol{\epsilon}$, so
the natural parameterization lets the network predict the noise,

$$
\boldsymbol{\mu}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \boldsymbol{\epsilon}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) \right),
$$

and the equation becomes a regression of
$\boldsymbol{\epsilon}_{\boldsymbol{\theta}}$ onto $\boldsymbol{\epsilon}$
with a step-dependent weight:

$$
L_{t-1} = \mathbb{E}_{\mathbf{x}_0, \boldsymbol{\epsilon}}\!\left[ \frac{\beta_t^2}{2 \sigma_t^2 \alpha_t (1 - \bar{\alpha}_t)} \big\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_{\boldsymbol{\theta}}\big(\sqrt{\bar{\alpha}_t}\, \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\, \boldsymbol{\epsilon},\ t\big) \big\|^2 \right] + \textrm{const}.
$$

For $\sigma_t^2 = \beta_t$ and the schedule used in the code below, the
weight in front is about one half at the start of the chain, falls roughly
like $1 / t$ to a tenth of that by $t = 20$, and stays between about $0.025$
and $0.06$ over the rest of the chain; $\sigma_t^2 = \tilde{\beta}_t$ makes
the early weights larger still (Exercise 3 traces both).
@ho2020denoising drop the weight and also drop the separate
treatment of $L_0$, training instead on the **simple loss**

$$
L_{\textrm{simple}}(\boldsymbol{\theta}) = \mathbb{E}_{t \sim \mathcal{U}\{1, \ldots, T\},\ \mathbf{x}_0,\ \boldsymbol{\epsilon}}\!\left[ \big\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_{\boldsymbol{\theta}}\big(\sqrt{\bar{\alpha}_t}\, \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\, \boldsymbol{\epsilon},\ t\big) \big\|^2 \right],
$$

a reweighting of the bound that emphasizes the noisier steps, where
recovering $\mathbf{x}_0$ from $\mathbf{x}_t$ is hardest even though
predicting $\boldsymbol{\epsilon}$ is easiest. @ho2020denoising
found that it gives better samples than the bound itself. The resulting
procedures are short.

**Training.** Repeat until converged: draw a data point $\mathbf{x}_0$, a
step $t$ uniformly from $\{1, \ldots, T\}$, and $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, I)$;
form $\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\, \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\, \boldsymbol{\epsilon}$;
take a gradient step on $\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_{\boldsymbol{\theta}}(\mathbf{x}_t, t)\|^2$.

**Sampling.** Draw $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, I)$. For
$t = T, \ldots, 1$: compute $\boldsymbol{\mu}_{\boldsymbol{\theta}}(\mathbf{x}_t, t)$
from the equation, and set
$\mathbf{x}_{t-1} = \boldsymbol{\mu}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) + \sigma_t \mathbf{z}$
with $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, I)$ for $t > 1$ and $\mathbf{z} = \mathbf{0}$
for $t = 1$. Return $\mathbf{x}_0$. The code below uses $\sigma_t^2 = \beta_t$.
This procedure, which draws the root of the chain from the prior and then
each variable from the learned conditional given its parent, is called
*ancestral sampling*.

### The Connection to Score Matching

The regression the equation is denoising score
matching. The conditional score of the marginal
the equation is
$\nabla_{\mathbf{x}_t} \log q(\mathbf{x}_t \mid \mathbf{x}_0) = -\boldsymbol{\epsilon} / \sqrt{1 - \bar{\alpha}_t}$,
so a noise predictor and a score model at step $t$ differ by a constant
factor,

$$
\mathbf{s}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) = -\frac{\boldsymbol{\epsilon}_{\boldsymbol{\theta}}(\mathbf{x}_t, t)}{\sqrt{1 - \bar{\alpha}_t}},
$$

and the simple loss equals twice the noise-conditional denoising objective
the equation with weights $\lambda_t = 1 - \bar{\alpha}_t$,
applied to the scaled data $\sqrt{\bar{\alpha}_t}\, \mathbf{x}_0$ at the
noise levels $\sqrt{1 - \bar{\alpha}_t}$, which play the role of the
$\sigma_i$ of that objective (that section gives
the two-line computation; the appendix writes the objective without the
factor $\tfrac12$, so it states equality). By
Vincent's theorem the minimizer of the equation over an
unrestricted function class is the conditional expectation
$\boldsymbol{\epsilon}^\star(\mathbf{x}_t, t) = \mathbb{E}[\boldsymbol{\epsilon} \mid \mathbf{x}_t]
= -\sqrt{1 - \bar{\alpha}_t}\, \nabla_{\mathbf{x}_t} \log q(\mathbf{x}_t)$,
where $q(\mathbf{x}_t)$ is the marginal of the forward process at step $t$.
The sampler is a score-based sampler as well. Substituting
the equation into the equation, one
reverse step reads

$$
\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}} \Big( \mathbf{x}_t + \beta_t\, \mathbf{s}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) \Big) + \sigma_t \mathbf{z},
$$

a step of length $\beta_t$ along the score, twice the drift that Langevin
dynamics would use with noise variance $\beta_t$, followed by Gaussian
noise, with a rescaling by $1 / \sqrt{\alpha_t}$ that undoes the forward
shrinkage. It resembles the annealed Langevin update of
that section, with a step size and a noise level per
step that the forward process dictates rather than the user. To first order
in $\beta_t$, half of the score step together with the noise is a Langevin
step that leaves $q(\mathbf{x}_t)$ unchanged, and the other half together
with the rescaling is a step of the probability-flow equation below, which
carries $q(\mathbf{x}_t)$ to $q(\mathbf{x}_{t-1})$. @song2021score
identify the whole step as a discretization of the time-reversed diffusion,
with Langevin steps as an optional corrector. The two lines of work that
this chapter has followed, the variational one from @sohl2015deep
and the score-based one from @song2019generative, arrive at the same
kind of model and differ in the forward process: the ladder adds noise
without shrinking the data, and the chain shrinks the data as it adds noise
[@song2021score; @Luo.2022].

The continuous-time view makes the identification exact, and it supplies
the vocabulary of the literature. Let the number of steps grow with
$\beta_t = \beta(t/T)/T$ for a rate function $\beta(\cdot)$ on $[0, 1]$, and
write $t$ from now on for the continuous time $t/T \in [0, 1]$. The forward
chain the equation then becomes the *variance-preserving*
stochastic differential equation

$$
d\mathbf{x} = -\tfrac{1}{2} \beta(t)\, \mathbf{x}\, dt + \sqrt{\beta(t)}\, d\mathbf{w},
$$

with $\mathbf{w}$ a Brownian motion, whereas the noise ladder of
that section, which adds noise without shrinking, is the
*variance-exploding* equation $d\mathbf{x} = \sqrt{d\sigma^2(t)/dt}\, d\mathbf{w}$,
with $\sigma(t)$ the ladder's noise level as a function of time.
Reversing time turns a diffusion into another diffusion whose drift
contains the score of the marginal [@Anderson.1982],

$$
d\mathbf{x} = \big[ -\tfrac{1}{2} \beta(t)\, \mathbf{x} - \beta(t)\, \nabla_{\mathbf{x}} \log q_t(\mathbf{x}) \big]\, dt + \sqrt{\beta(t)}\, d\bar{\mathbf{w}},
$$

run from $t = 1$ down to $0$ with $\bar{\mathbf{w}}$ a Brownian motion in
reversed time; the ancestral step the equation is a
discretization of it. The same marginals are also produced by an ordinary
differential equation, the *probability-flow* equation

$$
d\mathbf{x} = \big[ -\tfrac{1}{2} \beta(t)\, \mathbf{x} - \tfrac{1}{2} \beta(t)\, \nabla_{\mathbf{x}} \log q_t(\mathbf{x}) \big]\, dt,
$$

which carries half the score term and no noise. that section
meets a discretization of this equation.
that section identifies the chain with the
variance-preserving equation, and that section and
that section derive the reverse-time and the
probability-flow equations [@song2021score].

## Experiments on the Running Example

The network below predicts the noise from the current point and the step.
It is the score network of the earlier sections with a learned embedding of
the step index added to its first hidden layer; for $T = 200$ discrete steps
an embedding table is the simplest choice, and the sinusoidal features of
that section serve the same purpose for the image model
of the next section. Training implements the procedure above with the
simple loss.

In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.embed = nn.Embedding(T, hidden)  # one learned vector per step
        self.inp = nn.Linear(2, hidden)
        self.net = nn.Sequential(nn.SiLU(), nn.Linear(hidden, hidden),
                                 nn.SiLU(), nn.Linear(hidden, 2))

    def forward(self, x, t):
        return self.net(self.inp(x) + self.embed(t))

def train_ddpm(data, steps=4000, lr=1e-3, seed=2):
    torch.manual_seed(seed)
    net = NoisePredictor()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    for step in range(steps):
        x0 = data[torch.randint(0, len(data), (256,))]
        t = torch.randint(0, T, (256,))
        eps = torch.randn_like(x0)
        x_t = (alpha_bar[t].sqrt()[:, None] * x0
               + (1 - alpha_bar[t]).sqrt()[:, None] * eps)
        loss = ((net(x_t, t) - eps) ** 2).sum(1).mean()  # the simple loss
        optimizer.zero_grad(), loss.backward(), optimizer.step()
    return net

net = train_ddpm(data)

### The Learned Predictor against the Optimal One

For the running example the optimal noise predictor is available in closed
form: the marginal $q(\mathbf{x}_t)$ is the mixture with its means scaled by
$\sqrt{\bar{\alpha}_t}$, its component variances scaled by $\bar{\alpha}_t$,
and $1 - \bar{\alpha}_t$ added to each variance, which is what
`GaussianMixture.score` computes with the `scale` and
`sigma` arguments, and $\boldsymbol{\epsilon}^\star = -\sqrt{1 - \bar{\alpha}_t}$
times that score. The cell compares the network with
$\boldsymbol{\epsilon}^\star$ at four steps on noisy test points. It also
reports the irreducible part of the loss at each step,
$\mathbb{E}\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}^\star\|^2$, the
variance of the noise that cannot be inferred from $\mathbf{x}_t$, next to
the loss the network attains.

In [ ]:
def eps_star(x_t, t):  # the optimal noise predictor at step index t
    a = alpha_bar[t]
    return -(1 - a).sqrt() * mix.score(x_t, scale=a.sqrt().item(),
                                       sigma=(1 - a).sqrt().item())

torch.manual_seed(3)
print(f'{"step":>4} {"E||eps_theta - eps*||^2":>24} {"E||eps - eps*||^2":>18} '
      f'{"E||eps - eps_theta||^2":>23}')
with torch.no_grad():
    for t in (9, 49, 99, 199):
        eps = torch.randn_like(test)
        x_t = alpha_bar[t].sqrt() * test + (1 - alpha_bar[t]).sqrt() * eps
        pred, best = net(x_t, torch.full((len(test),), t)), eps_star(x_t, t)
        print(f'{t + 1:>4} {((pred - best) ** 2).sum(1).mean():>24.3f} '
              f'{((eps - best) ** 2).sum(1).mean():>18.3f} '
              f'{((eps - pred) ** 2).sum(1).mean():>23.3f}')

The mean squared difference between the learned and the optimal predictor
is at most a few hundredths at each step checked, on a target with unit
variance per coordinate. The second and third columns nearly coincide: the loss the network attains exceeds the
irreducible part by less than a tenth, and that part is largest at early
steps, where the noise is a
small part of $\mathbf{x}_t$ and hard to separate from the data, and falls
nearly to zero at the last step, where $\mathbf{x}_t$ is almost pure noise and the
noise is read off directly. Relative to the bound, the simple loss puts
more weight on these late steps, where predicting the noise is easiest.

### Ancestral Sampling

The sampler implements the procedure above. We run it with the learned
predictor and, for reference, with $\boldsymbol{\epsilon}^\star$, and
summarize the samples as in the previous sections.

In [ ]:
def ancestral_sample(eps_model, n, snapshot_steps=()):
    """The reverse chain from x_T ~ N(0, I) down to x_0."""
    x, snapshots = torch.randn(n, 2), {}
    with torch.no_grad():
        for t in reversed(range(T)):
            if t + 1 in snapshot_steps:
                snapshots[t + 1] = x.clone()
            eps = eps_model(x, torch.full((n,), t))
            mean = (x - beta[t] / (1 - alpha_bar[t]).sqrt() * eps) / alpha[t].sqrt()
            x = mean + (beta[t].sqrt() * torch.randn_like(x) if t > 0 else 0)
    snapshots[0] = x
    return x, snapshots

def summarize(x):
    d2 = ((x[:, None, :] - mix.means[None]) ** 2).sum(-1)
    weights = [round(w, 3) for w in
               (torch.bincount(d2.argmin(1), minlength=3) / len(x)).tolist()]
    nearest = d2.min(1).values
    inside = nearest < (3 * mix.std) ** 2
    return (f'mode weights {weights}, spread {nearest[inside].mean():.2f}, '
            f'stray {1 - inside.float().mean():.3f}')

torch.manual_seed(4)
print(f'target (2000 samples of the mixture): {summarize(data)}')
samples, snapshots = ancestral_sample(net, 2000, snapshot_steps=(200, 100, 50))
print(f'learned predictor: {summarize(samples)}')
exact, _ = ancestral_sample(lambda x, t: eps_star(x, t[0]), 2000)
print(f'optimal predictor: {summarize(exact)}')

Two hundred steps from pure noise reproduce the three modes: with the
optimal predictor the weights come within about two hundredths of the true
ones and the within-mode spread and the stray fraction match those of exact
samples to within sampling error, and with the learned predictor the weights
come within about five hundredths. The sampler has no parameters of its
own: its step sizes and noise levels are those of the forward schedule. The
panels below show the state of the chain at four points on the way down,
for the mixture and for a second data set, a noisy spiral, whose structure
is not a set of blobs. In both cases the chain starts as a Gaussian cloud
and stays one through the first half of the schedule, because the signal in
$\mathbf{x}_t$ is still small there. The mixture's three blobs have formed
by $t = 50$, whereas the spiral is still diffuse there and takes shape only
in the last fifty steps, which also sharpen both.

In [ ]:
torch.manual_seed(5)
u = torch.sqrt(torch.rand(2000)) * 3 * math.pi + 0.5
spiral = torch.stack([u * torch.cos(u), u * torch.sin(u)], 1) / 3
spiral = spiral + 0.1 * torch.randn(2000, 2)
net_spiral = train_ddpm(spiral, steps=6000)
_, snapshots_spiral = ancestral_sample(net_spiral, 2000,
                                       snapshot_steps=(200, 100, 50))
fig, axes = d2l.plt.subplots(2, 4, figsize=(12, 6))
for row, snaps in zip(axes, (snapshots, snapshots_spiral)):
    for ax, step in zip(row, (200, 100, 50, 0)):
        ax.scatter(snaps[step][:, 0], snaps[step][:, 1], s=2)
        ax.set_xlim(-5, 5), ax.set_ylim(-5, 5), ax.set_aspect('equal')
        ax.set_title(f't = {step}')
fig.tight_layout()

The model of this section trains by a regression with a closed-form target
instead of a game, and it rests on a likelihood bound; these answer two of
the difficulties that that section listed for adversarial
training. What it costs is the length of the reverse chain: two
hundred network evaluations here, a thousand in the original image models,
each a full forward pass. Reducing this cost without retraining is the
subject of that section. The next section scales the model
to images.

## Summary

A denoising diffusion probabilistic model fixes a forward Markov chain
the equation that shrinks the data and adds Gaussian noise
in $T$ steps, with the closed-form marginal the equation
that lets any step be sampled directly and that ends close to a standard
Gaussian.
The reverse chain is a latent-variable model with Gaussian transitions
whose means a network predicts. The variational bound on the likelihood
decomposes into a prior term, a reconstruction term, and one
Kullback--Leibler term per step the equation, each
comparing the learned transition with the Gaussian posterior
$q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)$ of
the equation.

Parameterizing the transition mean through a noise prediction turns every
step's term into a weighted regression of the predicted onto the actual
noise, and the simple loss the equation drops the
weights. This loss is denoising score matching across the noise levels of
the forward process, the optimal predictor is a scaled score of the
marginal, and one reverse step resembles a Langevin update with a schedule
set by the forward chain and a doubled drift: half of the score step and the
noise form a Langevin step at the current level, and the other half with the
rescaling carries the chain, to first order, to the next marginal. On the running example,
the learned predictor matched the closed-form optimum to within a mean
squared error of a few hundredths, its loss exceeded the irreducible noise
variance by less than a tenth, and ancestral sampling recovered the mixture
with no sampler parameters beyond the forward schedule and the choice
$\sigma_t^2 = \beta_t$.

## Exercises

1. **The marginal by induction.** Prove the equation by
   induction on $t$, using that a sum of independent Gaussians is Gaussian
   with summed variances, and verify the variance-preserving property: if
   $\mathbf{x}_0$ has identity covariance, so does every $\mathbf{x}_t$. Then
   compute $\bar{\alpha}_T$ for the schedule of @ho2020denoising
   ($T = 1000$, $\beta_t$ linear from $10^{-4}$ to $0.02$), approximating the
   product by $\exp(-\sum_t \beta_t)$, and compare with the value of
   $\bar{\alpha}_T$ in the code.
1. **The forward posterior.** Multiply the two Gaussian factors of
   $q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0)$, complete the square
   in $\mathbf{x}_{t-1}$, and derive the equation. Then
   substitute $\mathbf{x}_0 = (\mathbf{x}_t - \sqrt{1 - \bar{\alpha}_t}\, \boldsymbol{\epsilon}) / \sqrt{\bar{\alpha}_t}$
   and show that the coefficient of $\mathbf{x}_t$ in the posterior mean
   collapses to $1 / \sqrt{\alpha_t}$, which gives
   the equation.
1. [code] **The weights of the bound.** Compute the weight
   $\beta_t^2 / (2 \sigma_t^2 \alpha_t (1 - \bar{\alpha}_t))$ in
   the equation for $\sigma_t^2 = \beta_t$ and for
   $\sigma_t^2 = \tilde{\beta}_t$, as a function of $t = 2, \ldots, T$ for
   the schedule in the code, and plot both. Which steps does the bound emphasize relative to
   the simple loss, and how does this relate to the irreducible loss measured
   in the experiment?
1. **Two derivations, one model.** Starting from the equation,
   show that the simple loss equals twice the noise-conditional denoising
   objective the equation evaluated on the scaled data
   $\sqrt{\bar{\alpha}_t}\, \mathbf{x}_0$ at the noise levels of the forward
   process, with weights $\lambda_t = 1 - \bar{\alpha}_t$. Then
   substitute the equation into the equation
   to obtain the equation, and identify the step
   size and the noise scale of the corresponding Langevin update.
1. [code] **The prior term.** Estimate $L_T$ for the running example by
   computing the Kullback--Leibler divergence between
   $q(\mathbf{x}_T \mid \mathbf{x}_0)$ and $\mathcal{N}(\mathbf{0}, I)$ with
   the equation, averaged over training points. Repeat with
   $T = 50$ and the same $\beta$ range. How large does the prior term become,
   and what does a large value imply for samples that start from a standard
   Gaussian?
1. [code] **Sampling with fewer steps.** Run the reverse chain using only
   every fourth step of the schedule, treating the skipped steps as merged
   into one transition with the product of their $\alpha$ values, and
   compare the summary statistics with the full chain. Then try every tenth
   step. Where does the approximation break down first: in the mode weights,
   in the spread, or in the stray fraction?
1. [code] **Predicting the mean instead of the noise.** Train a network to
   output $\boldsymbol{\mu}_{\boldsymbol{\theta}}$ directly on the objective
   the equation with $\sigma_t^2 = \beta_t$, using the
   same architecture and number of updates, and compare the sampling
   statistics with the noise-prediction model. Explain the difference in the
   scale of the two regression targets across $t$.